# For geothermal Vienna basin, this notebook has been adjusted for the following: 
- Geomechanical simulations are stored in sr3 files, not in separate gmch.sr3 files.

# File format conversion
CMG sr3/gmch.sr3 --> CMG rwo (by running CMG Results Report software on rwd files) --> numpy array  
sr3 is the binary CMG simulation result file, whereas rwo is the extracted info in the ascii format  
rwd is a configuration file to tell CMG Results Report software how to export results

# Step 1: Convert CMG sr3/gmch.sr3 files to rwo files

Note this part needs to be run on a machine where CMG Results Report software is installed and the CMG2npy source code file.

In [1]:
from pathlib import Path
from CMG2npy import generate_CMG_rwd, run_CMG_rwd_report
from tqdm import tqdm

current_path = Path('.')

################## User Inputs ############################## 
name_prefix = 'test_260703'; n_cases = 10
property_list = ['STRESMXP','STRESMNP','STRESINT']
sim_results_folder_path = current_path/f'{name_prefix}_dat_files' # path to the simulation results folder
sim_results_file_format = 'sr3' # sr3 or gmch.sr3
################## End of User Inputs #######################

for property in property_list:
    for case_num in tqdm(range(1,n_cases+1),desc=f'Converting sr3 to rwo for keyword {property}'):
        generate_CMG_rwd(
            sr3_folder_path = sim_results_folder_path,
            case_name = f'case{case_num}',
            property = property,
            sim_results_file_format = sim_results_file_format,
            precision = 4
        )

        run_CMG_rwd_report(
            rwd_folder_path = sim_results_folder_path,
            case_name = f'case{case_num}',
            cmg_version = 'ese-ts2win-v2024.20',
        )

        # remove rwd files
        Path(sim_results_folder_path, f'case{case_num}.rwd').unlink()

print("\nFinished generating rwo files for all cases.")

Converting sr3 to rwo for keyword STRESINT: 100%|██████████| 10/10 [00:31<00:00,  3.12s/it]


Finished generating rwo files for all cases.


# Step 2: Extract simulation results from rwo files into numpy arrays

## Option 1: Extract results on all grid cells

In [4]:
from pathlib import Path
from CMG2npy import CMG_rwo2npy
from tqdm import tqdm

current_path = Path('.')

################## User Inputs ############################## 
name_prefix = 'test_260703'; n_cases = 10
property_list = ['STRESMXP','STRESMNP','STRESINT']
sim_results_folder_path = current_path/f'{name_prefix}_dat_files' # path to the simulation results folder
save_folder_path = sim_results_folder_path/f'{name_prefix}_processed' # path to save processed results
################## End of User Inputs #######################

save_folder_path.mkdir(parents=True, exist_ok=True)

for property in property_list:
    for case_num in tqdm(range(1,n_cases+1), desc=f'Generating numpy arrays for keyword {property}'):
        sim_results = CMG_rwo2npy(
            rwo_folder_path = sim_results_folder_path/'rwo',
            case_name = f'case{case_num}',
            property = property,
            is_save = True,
            save_folder_path = save_folder_path,
            show_info = False
        )

print("\nFinished generating numpy arrays for all cases.")

Generating numpy arrays for keyword STRESINT: 100%|██████████| 10/10 [00:13<00:00,  1.30s/it]


Finished generating numpy arrays for all cases.


## Option 2: Only keep fault grid cells and simulated layers

- For the JD_geothermal grid (139, 248, 23), reservoir is between k=5 and 20.
- Note this part needs the file containing grid cell coordinates and fault id.

In [2]:
import numpy as np
from pathlib import Path
from CMG2npy import CMG_rwo2npy
from tqdm import tqdm

current_path = Path('.')

################## User Inputs ############################## 
name_prefix = 'test_260703'; n_cases = 10
property_list = ['STRESMXP','STRESMNP','STRESINT']
sim_results_folder_path = current_path/f'{name_prefix}_dat_files' # path to the simulation results folder
coor_fault_file_path = current_path/'JD_geothermal_coor&fault.npy' # path to the file containing grid cell coordinates and fault id
save_folder_path = sim_results_folder_path/f'{name_prefix}_gmc' # path to save processed results
################## End of User Inputs #######################

coor_fault = np.load(coor_fault_file_path) 
save_folder_path.mkdir(parents=True, exist_ok=True)

for property in property_list:
    for case_num in tqdm(range(1,n_cases+1),desc=f'Generating numpy arrays for keyword {property}'):
        # extract results on all grid cells
        sim_results = CMG_rwo2npy(
            rwo_folder_path = sim_results_folder_path/'rwo',
            case_name = f'case{case_num}',
            property = property,
            is_save = False,
            save_folder_path = save_folder_path,
            show_info = False
        )

        # keep results on fault grid cells only
        nan_mask = np.isnan(coor_fault[:,:,:,3])
        sim_results[nan_mask] = np.nan

        # keep results of simulated layers only
        # Python indices are 0-based, to extract slices 41–79, use 40:79
        sim_results_trimmed = sim_results[:, :, 5:10]

        # save
        np.save(save_folder_path/f'case{case_num}_{property}.npy',sim_results_trimmed)


print("\nFinished generating numpy arrays for all cases.")

Generating numpy arrays for keyword STRESINT: 100%|██████████| 10/10 [00:13<00:00,  1.35s/it]


Finished generating numpy arrays for all cases.
